# Runnable Retries

The `retry.py` module defines a serializable Runnable binding that retries a wrapped Runnable when selected exception types are raised.

Retry behaviour is implemented through Tenacity. It supports synchronous and asynchronous invocation, exponential backoff with optional jitter, configurable attempt limits, and selective retries for failed batch elements.

# ExponentialJitterParams

`ExponentialJitterParams` defines optional parameters passed to Tenacity's `wait_exponential_jitter` strategy.

Every field is optional.

## Bases

- `TypedDict`

## Attributes

1. `initial`: Stores the initial waiting interval before exponential growth is applied.
   * **Type:**
     ```python
     initial: float
     ```

2. `max`: Stores the maximum waiting interval.
   * **Type:**
     ```python
     max: float
     ```

3. `exp_base`: Stores the base used to calculate exponential backoff.
   * **Type:**
     ```python
     exp_base: float
     ```

4. `jitter`: Stores the upper bound of the additional random waiting interval.

   The additional delay is sampled uniformly between zero and this value.

   * **Type:**
     ```python
     jitter: float
     ```

# RunnableRetry

`RunnableRetry` wraps another Runnable and retries its execution when a configured exception type is raised.

It is implemented as a `RunnableBindingBase`. The wrapper can be created directly or through the `with_retry` method inherited by Runnables.

Retries should generally be applied only to the smallest Runnable operation likely to fail, rather than to an entire chain containing non-idempotent operations.

## Bases

- `RunnableBindingBase[Input, Output]`

## Attributes

1. `retry_exception_types`: Stores the exception types that activate retry behaviour.

   By default, every `Exception` subclass is retried. Exceptions outside this tuple are raised without retrying.

   * **Type:**
     ```python
     retry_exception_types: tuple[
         type[BaseException],
         ...
     ] = (Exception,)
     ```

2. `wait_exponential_jitter`: Controls whether exponential backoff with random jitter is used between attempts.

   When disabled, no waiting strategy is added by this class.

   * **Type:**
     ```python
     wait_exponential_jitter: bool = True
     ```

3. `exponential_jitter_params`: Stores optional custom values for the exponential-jitter waiting strategy.

   These values are used only when `wait_exponential_jitter` is enabled.

   * **Type:**
     ```python
     exponential_jitter_params: ExponentialJitterParams | None = None
     ```

4. `max_attempt_number`: Stores the maximum total number of execution attempts.

   The count includes the initial attempt. Its default value therefore permits one initial attempt and up to two additional attempts.

   * **Type:**
     ```python
     max_attempt_number: int = 3
     ```

### Methods

1. `invoke`: Executes the wrapped Runnable synchronously and retries it when a configured exception is raised.

   Every retry receives a child callback manager. Attempts after the first are tagged using the format `retry:attempt:<attempt_number>`.

   When all configured attempts fail, the final exception is re-raised by Tenacity.

   * **Syntax:**
     ```python
     invoke(
         self,
         input: Input, # Input passed to the wrapped Runnable
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional invocation arguments
     ) -> Output
     ```

2. `ainvoke`: Executes the wrapped Runnable asynchronously and retries it when a configured exception is raised.

   It follows the same attempt limit, exception filtering, waiting strategy, and callback-tagging behaviour as `invoke`.

   When all configured attempts fail, the final exception is re-raised by Tenacity.

   * **Syntax:**
     ```python
     async ainvoke(
         self,
         input: Input, # Input passed to the wrapped Runnable
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional invocation arguments
     ) -> Output
     ```

3. `batch`: Executes multiple inputs synchronously with retry support.

   The wrapped Runnable is called with `return_exceptions=True` internally so successful and failed elements can be separated. Successful elements are preserved and are not executed again. Only inputs that have not yet succeeded are passed to later attempts.

   Results are restored to the same order as the original inputs. When `return_exceptions` is `False`, remaining failures are handled by the inherited batch callback machinery. When it is `True`, exceptions may be returned in their corresponding result positions.

   * **Syntax:**
     ```python
     batch(
         self,
         inputs: list[Input], # Inputs passed to the wrapped Runnable
         config: RunnableConfig
         | list[RunnableConfig]
         | None = None, # Shared or per-input runtime configuration
         *,
         return_exceptions: bool = False, # Return exceptions instead of raising them
         **kwargs: Any # Additional batch arguments
     ) -> list[Output]
     ```

4. `abatch`: Executes multiple inputs asynchronously with retry support.

   Successful elements are preserved after each attempt, while only failed elements are included in subsequent asynchronous batch attempts. Output positions continue to correspond to the original input order.

   It follows the same exception-return and configuration rules as `batch`.

   * **Syntax:**
     ```python
     async abatch(
         self,
         inputs: list[Input], # Inputs passed to the wrapped Runnable
         config: RunnableConfig
         | list[RunnableConfig]
         | None = None, # Shared or per-input runtime configuration
         *,
         return_exceptions: bool = False, # Return exceptions instead of raising them
         **kwargs: Any # Additional asynchronous batch arguments
     ) -> list[Output]
     ```

## Retry Configuration Behaviour

The retry controller is constructed from the following settings:

- `max_attempt_number` produces a Tenacity `stop_after_attempt` rule.
- `wait_exponential_jitter=True` produces a `wait_exponential_jitter` rule using `exponential_jitter_params`.
- A non-empty `retry_exception_types` tuple produces a `retry_if_exception_type` rule.

Synchronous execution uses Tenacity's `Retrying`, while asynchronous execution uses `AsyncRetrying`.

## Streaming Limitation

`RunnableRetry` does not override `stream`, `astream`, `transform`, or `atransform`.

Streaming operations therefore use the inherited behaviour and are not retried by this class because restarting a partially consumed stream could duplicate or invalidate previously emitted output.